# NB9 — RF-DETR + FashionCLIP Core-7 Detection Smoke Test

Notebook này là smoke test **end-to-end** cho detection pipeline:

```text
input image
  -> RF-DETR garment boxes
  -> garment crops
  -> FashionCLIP 512-d L2 image embedding
  -> cosine similarity với 7 Core-7 text prototypes
  -> coarse_category
  -> scorer handoff tensors
```

`master_category` không được suy diễn cho ảnh user.

**PASS duy nhất:** cell cuối in chính xác `NB9 SMOKE PASS`.


## 1. Checkout branch + clear project module cache

Cell này checkout đúng feature branch và xóa toàn bộ `src.*` khỏi `sys.modules`.
Điểm này bắt buộc khi notebook pull code mới trong cùng Colab kernel; nếu không Python có thể tiếp tục chạy implementation cũ dù file trên disk đã thay đổi.


In [ ]:
from pathlib import Path
import importlib
import os
import shutil
import subprocess
import sys
import time

REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"
BRANCH = "feat/detection-rfdetr-fashionclip-core7"
REPO_DIR = Path("/content/opisoverated")

def run_git(args, *, cwd=None, check=True):
    cmd = ["git", "-c", "http.version=HTTP/1.1", *args]
    proc = subprocess.run(cmd, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if proc.stdout:
        print(proc.stdout, end="" if proc.stdout.endswith("\n") else "\n")
    if check and proc.returncode != 0:
        raise RuntimeError(f"Git command failed ({proc.returncode}): {' '.join(cmd)}\n{proc.stdout}")
    return proc

probe = None
for attempt in range(1, 4):
    print(f"[git preflight] attempt {attempt}/3")
    probe = run_git(["ls-remote", "--exit-code", "--heads", REPO_URL, f"refs/heads/{BRANCH}"], check=False)
    if probe.returncode == 0 and (probe.stdout or "").strip():
        break
    if attempt < 3:
        time.sleep(2 * attempt)
else:
    raise RuntimeError("Cannot reach the requested GitHub branch.\n" f"repo={REPO_URL}\nbranch={BRANCH}\n" f"git output:\n{probe.stdout if probe else ''}")

git_dir = REPO_DIR / ".git"
if git_dir.is_dir():
    run_git(["remote", "set-url", "origin", REPO_URL], cwd=REPO_DIR)
    run_git(["fetch", "--depth", "1", "origin", BRANCH], cwd=REPO_DIR)
    run_git(["checkout", "-B", BRANCH, "FETCH_HEAD"], cwd=REPO_DIR)
    run_git(["reset", "--hard", "FETCH_HEAD"], cwd=REPO_DIR)
    run_git(["clean", "-fdx"], cwd=REPO_DIR)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    for attempt in range(1, 4):
        print(f"[git clone] attempt {attempt}/3")
        clone = run_git(["clone", "--depth", "1", "--single-branch", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=False)
        if clone.returncode == 0:
            break
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        if attempt < 3:
            time.sleep(2 * attempt)
    else:
        raise RuntimeError(f"git clone failed after 3 attempts.\n{clone.stdout or ''}")

os.chdir(REPO_DIR)
stale_project_modules = [name for name in list(sys.modules) if name == "src" or name.startswith("src.")]
for name in stale_project_modules:
    sys.modules.pop(name, None)
importlib.invalidate_caches()
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=REPO_DIR, text=True).strip()
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
assert branch == BRANCH, (branch, BRANCH)
print("repo:", REPO_DIR)
print("branch:", branch)
print("commit:", commit)
print("cleared project modules:", len(stale_project_modules))
print("checkout/cache reset: PASS")


## 2. Install compatible runtime dependencies

Không dùng `--upgrade`. Nếu pip thay NumPy/SciPy trong khi package cũ đã được import trong kernel, notebook yêu cầu restart runtime thay vì tiếp tục với mixed binary state.


In [ ]:
from importlib import metadata as importlib_metadata
from packaging.version import Version
REQUIREMENTS_PATH = REPO_DIR / "requirements-detection.txt"
print(REQUIREMENTS_PATH.read_text(encoding="utf-8"))
binary_packages = ("numpy", "scipy")
loaded_before = {name: getattr(sys.modules.get(name), "__version__", None) for name in binary_packages if name in sys.modules}
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REQUIREMENTS_PATH)], check=True)
dist_after = {name: importlib_metadata.version(name) for name in binary_packages}
changed_loaded = {name: (loaded_before[name], dist_after[name]) for name in loaded_before if loaded_before[name] != dist_after[name]}
if changed_loaded:
    raise RuntimeError("A binary dependency changed on disk while its old version is already loaded in this kernel: " f"{changed_loaded}. Restart the Colab runtime, then Run all again.")
versions = {}
for name in ("torch", "numpy", "scipy", "transformers", "rfdetr", "huggingface_hub"):
    try:
        versions[name] = importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        versions[name] = None
assert Version(versions["numpy"]) < Version("2.4")
assert Version(versions["transformers"]) >= Version("5.1")
assert Version(versions["transformers"]) < Version("6")
for name, version in versions.items():
    print(f"{name}=={version}")
print("dependency contract: PASS")


## 3. Runtime preflight

Import NumPy/SciPy/RF-DETR/Transformers trước khi tải model để bắt sớm lỗi binary/runtime.


In [ ]:
import numpy as np
import scipy
import torch
import transformers
import rfdetr
import numpy.testing._private.utils as numpy_testing_utils
assert np.isfinite(np.array([0.0, 1.0])).all()
assert hasattr(numpy_testing_utils, "assert_allclose")
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("runtime preflight: PASS")


## 4. Import project code fresh + verify Transformers-v5 adapter

Cell này xóa `src.*` thêm một lần ngay trước project import, rồi kiểm tra source đang chạy thật sự có `_extract_feature_tensor`.
Nếu assertion này fail thì kernel đang giữ stale project code.


In [ ]:
import importlib
import inspect
for name in [name for name in list(sys.modules) if name == "src" or name.startswith("src.")]:
    sys.modules.pop(name, None)
importlib.invalidate_caches()
import src.detection.fashionclip as fashionclip_module
from src.detection import CORE7_CATEGORIES, CORE7_CATEGORY_TO_ID, DetectionPipeline, load_detection_config
source_text = inspect.getsource(fashionclip_module)
assert "_extract_feature_tensor" in source_text, ("Stale src.detection.fashionclip is loaded. Rerun the checkout/cache-reset cell or restart the runtime.")
print("fashionclip module:", fashionclip_module.__file__)
print("Transformers-v5 feature adapter present: PASS")


## 5. Lightweight contract tests


In [ ]:
test_run = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_detection_core7.py", "-v"], cwd=REPO_DIR, text=True)
if test_run.returncode != 0:
    raise RuntimeError("Detection contract tests failed.")
print("unit tests: PASS")


## 6. Load config + smoke image


In [ ]:
from PIL import Image
from IPython.display import display
CONFIG_PATH = REPO_DIR / "configs/detection_rfdetr_fashionclip_core7_v1.json"
IMAGE_PATH = REPO_DIR / "tests/animage.jpg"
OUTPUT_DIR = REPO_DIR / "outputs/nb9_detection_smoke"
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(CONFIG_PATH)
if not IMAGE_PATH.is_file():
    raise FileNotFoundError(IMAGE_PATH)
config = load_detection_config(CONFIG_PATH)
smoke_image = Image.open(IMAGE_PATH).convert("RGB")
print("config:", CONFIG_PATH)
print("image:", IMAGE_PATH)
print("image size:", smoke_image.size)
print("Core-7:", CORE7_CATEGORY_TO_ID)
display(smoke_image)


## 7. Run RF-DETR → FashionCLIP → Core-7

Lần chạy đầu có thể tải RF-DETR checkpoint và FashionCLIP weights.


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
pipeline = DetectionPipeline(config, device=device)
result, image = pipeline.run(IMAGE_PATH)
print("accepted garments:", len(result.garments))
print("rejected detections:", len(result.rejected_detections))
print("RF-DETR runtime ms:", result.detector_runtime_ms)
for index, garment in enumerate(result.garments):
    print(f"[{index}]", garment.candidate.detector_label, "->", garment.category.coarse_category, f"det_conf={garment.candidate.detector_confidence}", f"sim={garment.category.similarity:.4f}", f"margin={garment.category.margin:.4f}")
if not result.garments:
    raise RuntimeError("RF-DETR/FashionCLIP produced zero accepted garments.")


## 8. Validate embeddings + categories


In [ ]:
for index, garment in enumerate(result.garments):
    embedding = torch.as_tensor(garment.embedding, dtype=torch.float32)
    norm = float(torch.linalg.vector_norm(embedding))
    category = garment.category.coarse_category
    category_id = garment.category.coarse_category_id
    assert embedding.shape == (512,), (index, embedding.shape)
    assert bool(torch.isfinite(embedding).all()), index
    assert abs(norm - 1.0) < 1e-3, (index, norm)
    assert category in CORE7_CATEGORY_TO_ID, category
    assert category_id == CORE7_CATEGORY_TO_ID[category], (category, category_id)
    print(index, category, category_id, f"norm={norm:.6f}")
print("embedding/category contract: PASS")


## 9. Save outputs + validate scorer handoff


In [ ]:
import json
from src.detection.pipeline import save_detection_result
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
saved = save_detection_result(result, image, OUTPUT_DIR, scorer_min_items=config.scorer_min_items, scorer_max_items=config.scorer_max_items)
print(json.dumps(saved, indent=2))
if saved["scorer_handoff_error"]:
    raise RuntimeError(saved["scorer_handoff_error"])
metadata_path = Path(saved["metadata_path"])
scorer_path = Path(saved["scorer_inputs_path"])
assert metadata_path.is_file()
assert scorer_path.is_file()
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
scorer_batch = torch.load(scorer_path, map_location="cpu")
n = len(result.garments)
assert scorer_batch["item_embeddings"].shape == (1, n, 512)
assert scorer_batch["coarse_category_ids"].shape == (1, n)
assert scorer_batch["item_mask"].shape == (1, n)
assert metadata["taxonomy"]["master_category"] is None
for garment in metadata["garments"]:
    assert "master_category" not in garment
print("scorer handoff: PASS")


## 10. Final gate


In [ ]:
assert len(result.garments) >= config.scorer_min_items
assert len(result.garments) <= config.scorer_max_items
assert metadata_path.is_file()
assert scorer_path.is_file()
print("NB9 SMOKE PASS")
